# dispatch-back-fn-from-recipe — worked example 2: Dispatch back functions for a two-parent node with different gradients per argnum

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dispatch-back-fn-from-recipe`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a forward function takes multiple arguments (like division or subtraction), each argument may have a completely different gradient rule. The `(func, argnum)` pair uniquely identifies which rule to apply. The dispatch loop iterates all entries in `recipe.parents` and retrieves the matching back function for each — collecting one triple `(argnum, parent, back_fn)` per parent.

## Worked solution

Node `z` was produced by `sub(a, b)`. Its recipe: `func=sub`, `parents={0: a, 1: b}`.

The back functions are:
- `sub_back_0(grad, out, a, b) = grad` (gradient flows through the minuend unchanged).
- `sub_back_1(grad, out, a, b) = -grad` (gradient through the subtrahend is negated).

**Iteration:**
- `(argnum=0, parent=a)` → key `(sub, 0)` → `sub_back_0`. Triple: `(0, a, sub_back_0)`.
- `(argnum=1, parent=b)` → key `(sub, 1)` → `sub_back_1`. Triple: `(1, b, sub_back_1)`.

**Result.** Two triples, each carrying the exactly-correct gradient function. Notice `sub_back_0 ≠ sub_back_1` — asymmetry captured purely by the argnum key.

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Recipe:
    func: Callable
    parents: dict
    args: tuple = ()
    kwargs: dict = None

class FakeTensor:
    def __init__(self, name, recipe=None):
        self.name = name
        self.recipe = recipe
    def __repr__(self): return f'FakeTensor({self.name})'

def sub(a, b): return a - b
def sub_back_0(grad, out, a, b): return grad          # d/da (a-b) = 1
def sub_back_1(grad, out, a, b): return -grad         # d/db (a-b) = -1

a = FakeTensor('a')
b = FakeTensor('b')
z = FakeTensor('z', Recipe(func=sub, parents={0: a, 1: b}))

back_funcs = {
    (sub, 0): sub_back_0,
    (sub, 1): sub_back_1,
}

def dispatch_back_fns(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    return results

triples = dispatch_back_fns(z, back_funcs)
print('Triples:')
for argnum, parent, fn in triples:
    print(f'  argnum={argnum}, parent={parent}, fn={fn.__name__}')

# Verify back_fn correctness on sample gradient
grad_out = 3.0
for argnum, parent, fn in triples:
    result = fn(grad_out, z, a, b)
    expected = grad_out if argnum == 0 else -grad_out
    print(f'  argnum={argnum}: fn({grad_out}) = {result}, expected {expected}, correct: {result == expected}')